In [1]:
!pip install catboost ipywidgets graphviz numpy plotly six pandas scipy category_encoders matplotlib lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.1 MB/s eta 0:00:00


In [2]:
import random
random.seed(42)
import numpy as np
np.random.seed(42)

In [3]:
import pandas as pd
import numpy as np
from category_encoders.james_stein import JamesSteinEncoder
from category_encoders.cat_boost import CatBoostEncoder
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.base import BaseEstimator
from sklearn.base import TransformerMixin



class ValidatedStackedMeanEncoder(BaseEstimator, TransformerMixin):
    """
    repeated cross-validation in constructing mean-encoding
    """
    def __init__(self):
        """
        :param cols: Categorical columns
        :param encoders_names_tuple: Tuple of str with encoders
        """
        #self.target_encoder = JamesSteinEncoder
        self.target_encoder = CatBoostEncoder
        k_folds = 5
        n_repeats = 3

        self.k_folds, self.n_repeats = k_folds, n_repeats
        self.model_validation = RepeatedStratifiedKFold(n_splits=self.k_folds, n_repeats=self.n_repeats, random_state=0)
        self.list_of_encoders = []
        self._fit_index = None

    def fit(self, X: pd.DataFrame, y: pd.Series):
        X = pd.DataFrame(X).reset_index(drop=True)
        y = pd.Series(y).reset_index(drop=True)
        self._fit_index = X.index
        self.list_of_encoders=[]

        for n_fold, (train_idx, val_idx) in enumerate(self.model_validation.split(X, y)):
            encoder = self.target_encoder(cols=X.columns)
            X_train = X.loc[train_idx, :]
            y_train = y.loc[train_idx]
            encoder.fit(X_train, y_train)
            self.list_of_encoders.append(encoder)

        return self

    def fit_transform(self, X, y):
        X = pd.DataFrame(X).reset_index(drop=True)
        y = pd.Series(y).reset_index(drop=True)
        cols_representation = np.zeros_like(X, dtype=float)
        cols = X.columns
        self.list_of_encoders = []
        for n_fold, (train_idx, val_idx) in enumerate(self.model_validation.split(X, y)):
            print('in fit_transform')

            encoder = self.target_encoder(cols=X.columns)
            X_train, X_val = X.loc[train_idx, :], X.loc[val_idx, :]
            y_train, y_val = y[train_idx], y[val_idx]
            encoder.fit(X_train, y_train)

            # transform validation part and get all necessary cols
            val_t = encoder.transform(X_val)

            cols_representation[val_idx, :] += val_t / self.n_repeats
            self.list_of_encoders.append(encoder)

        cols_representation = pd.DataFrame(cols_representation, columns=['mean_enc_' + c for c in cols])
        X = pd.concat([X, cols_representation], axis=1)
        return X

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = pd.DataFrame(X).reset_index(drop=True)
        cols_representation = np.zeros_like(X, dtype=float)
        cols = X.columns
        if self._fit_index and len(self._fit_index) == len(X.index):
            print('in if of transform')
            for n_fold, (train_idx, val_idx) in enumerate(self.model_validation.split(X)):
                val_t = self.list_of_encoders[n_fold].transform(X.loc[val_idx].reset_index(drop=True))
                cols_representation[val_idx, :] += val_t / self.n_repeats
        else:
            print('in else of transform')

            for encoder in self.list_of_encoders:
                test_tr = encoder.transform(X)
                cols_representation += test_tr / (self.k_folds * self.n_repeats)

        cols_representation = pd.DataFrame(cols_representation.values, columns=['mean_enc_' + c for c in cols])
        X = pd.concat([X, cols_representation], axis=1)
        # X.drop(cols, axis=1, inplace=True)
        return X


## This is the data-set, catboost became famous for:

In [4]:
from catboost.datasets import amazon
employee_train, employee_test = amazon()

The data is taken from a kaggle competition where catboost was shining:<br>
[https://www.kaggle.com/c/amazon-employee-access-challenge](https://www.kaggle.com/c/amazon-employee-access-challenge)

In [5]:
import warnings
import pandas as pd
from sklearn.model_selection import train_test_split

In [6]:
y = employee_train['ACTION']
X = employee_train.drop('ACTION', axis=1)

# Split into train & validation set
X_train, X_val, y_train, y_val = train_test_split(X, y, train_size=0.8)

### catboost

[2018: catboost the new kid on the block from russia](https://arxiv.org/pdf/1810.11363.pdf)

In [7]:
from google.colab import output
output.enable_custom_widget_manager()

In [8]:
from catboost import CatBoostClassifier
# import ipywidgets
cat_features = [*range(8)]
model = CatBoostClassifier(custom_metric=['TotalF1'], early_stopping_rounds=100, eval_metric='AUC')

model.fit(X_train, y_train, cat_features=cat_features,
          eval_set=(X_val, y_val), plot=True, verbose=10, use_best_model=True)


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Learning rate set to 0.071005
0:	test: 0.4996350	best: 0.4996350 (0)	total: 76.5ms	remaining: 1m 16s
10:	test: 0.7330358	best: 0.7330358 (9)	total: 437ms	remaining: 39.3s
20:	test: 0.8410509	best: 0.8410509 (20)	total: 902ms	remaining: 42s
30:	test: 0.8517744	best: 0.8517744 (30)	total: 1.37s	remaining: 42.9s
40:	test: 0.8725299	best: 0.8725299 (40)	total: 1.84s	remaining: 43s
50:	test: 0.8783833	best: 0.8783833 (50)	total: 2.36s	remaining: 43.9s
60:	test: 0.8849430	best: 0.8850609 (59)	total: 2.87s	remaining: 44.2s
70:	test: 0.8869148	best: 0.8869148 (70)	total: 3.46s	remaining: 45.3s
80:	test: 0.8883129	best: 0.8883489 (79)	total: 4.03s	remaining: 45.7s
90:	test: 0.8902088	best: 0.8902088 (90)	total: 4.56s	remaining: 45.6s
100:	test: 0.8917917	best: 0.8917917 (100)	total: 5.15s	remaining: 45.8s
110:	test: 0.8920255	best: 0.8920491 (109)	total: 5.7s	remaining: 45.7s
120:	test: 0.8922656	best: 0.8922656 (120)	total: 6.2s	remaining: 45s
130:	test: 0.8930898	best: 0.8930898 (130)	total: 

### lightgbm in comparison

Remark, that for catboost, we did no hyperparameter tuning at all.<br>

Now, let's try lightgbm:

__BUT__: since catboost has a early-stopping mechanism, we give the lightgbm-algorithm an eval_set and specify the 'early_stopping_rounds'. When the score on the validation-set is not becoming better than the best current score for the next 'early_stopping_rounds'-iterations, then the algorithm stops and returns the currently best result. We evaluate as well on the trainings-set to get a feeling for the degree of overfitting.

In [9]:
import lightgbm
model = lightgbm.LGBMClassifier(metric='auc', n_estimators=5000, learning_rate=0.02, random_state=42, verbose=100,early_stopping_rounds=100)
model.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_val, y_val)])

[LightGBM] [Warning] early_stopping_round is set=100, early_stopping_rounds=100 will be ignored. Current value: early_stopping_round=100
[LightGBM] [Info] Number of positive: 24687, number of negative: 1528
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.000047
[LightGBM] [Debug] init for col-wise cost 0.000434 seconds, init for row-wise cost 0.004130 seconds
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005410 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1793
[LightGBM] [Info] Number of data points in the train set: 26215, number of used features: 9
[LightGBM] [Warning] early_stopping_round is set=100, early_stopping_rounds=100 will be ignored. Current value: early_stopping_round=100
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.941713 -> initscore=2.782317
[LightGBM] [Info] Start training from score 2.782317
[LightGBM] [Debug] Trained a tree with leaves = 31 and dept

LGBMClassifier(early_stopping_rounds=100, learning_rate=0.02, metric='auc',
               n_estimators=5000, random_state=42, verbose=100)

## Now, we add the mean-encoding manually as a preprocessing step<br>

### this is how lightgbm could mimick catboost:

Since all variables are categorical, we add an OneHotEncoder. Since catboost became famous for its category-encoding, we add this as a second step. However, doing category-encoding always bears the risk of overfitting. This is why we do it via __stacking__.<br>
If we improve on the solution above it could also be because of the OneHotEncoder and not because of the ValidatedStackedMeanEncoder. Hence, we have to test the pipeline first with just the OneHotEncoder.

### without mean-encoding
The setup is the same, except that instead of the ValidatedStackedMeanEncoder, we pass FunctionTransformer(None) to the categorical variables - everything else is the same.<br>

__Attentions__: because we want to keep the early_stopping_round (in order to mimick catboost), we first have to one-hot-encode also the validation data set. Otherwise it would have a different number of columns.

In [10]:
import lightgbm
from sklearn.preprocessing import FunctionTransformer
import sys
import os
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
numerical = []
categorical = X_train.columns


model = lightgbm.LGBMClassifier(boosting_type='gbdt', num_leaves=31, max_depth=- 1, learning_rate=0.02,
                                n_estimators=5000, subsample_for_bin=20000, objective='binary',
                                subsample=1.0, subsample_freq=0, colsample_bytree=1.0,
                                n_jobs=- 1, silent=True, importance_type='split',
                                is_unbalance = False, scale_pos_weight = 1.0, random_state=42, metric='auc',
                                verbose=1, early_stopping_rounds=100
                              )

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(missing_values=np.nan, strategy='median')),
    ('scaler', StandardScaler())])

preprocessor1 = ColumnTransformer(transformers=[('num', numeric_transformer, numerical),
                                            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical),
                                            ('empty', FunctionTransformer(None), categorical)])
custom_pipeline1 = make_pipeline(
            preprocessor1,
            model
            )
X_train_preprocessed = preprocessor1.fit_transform(X_train)
X_val_preprocessed = preprocessor1.transform(X_val)
from sklearn.model_selection import StratifiedKFold
custom_pipeline1.fit(X_train, y_train, lgbmclassifier__eval_set=[(X_train_preprocessed, y_train), (X_val_preprocessed, y_val)])
#custom_pipeline.predict(test)
print("model score: %.3f" % custom_pipeline1.score(X_val, y_val))

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] Unknown parameter: silent
[LightGBM] [Warning] early_stopping_round is set=100, early_stopping_rounds=100 will be ignored. Current value: early_stopping_round=100
[LightGBM] [Warning] Unknown parameter: silent
[LightGBM] [Info] Number of positive: 24687, number of negative: 1528
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.141962 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4616
[LightGBM] [Info] Number of data points in the train set: 26215, number of used features: 1425
[LightGBM] [Warning] Unknown parameter: silent
[LightGBM] [Warning] early_stopping_round is set=100, early_stopping_rounds=100 will be ignored. Current value: early_stopping_round=100
[LightGBM] [Warning] Unknown parameter: silent
[LightGBM] [Warning] early_stopping_round is set=100, early_stopping_rounds=100 will be ignored. Current value: e

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] Unknown parameter: silent
model score: 0.952


In [11]:
from sklearn.metrics import classification_report, roc_auc_score
print(classification_report(y_val, custom_pipeline1.predict(X_val)))

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] Unknown parameter: silent
              precision    recall  f1-score   support

           0       0.68      0.28      0.40       369
           1       0.96      0.99      0.98      6185

    accuracy                           0.95      6554
   macro avg       0.82      0.64      0.69      6554
weighted avg       0.94      0.95      0.94      6554



In [12]:
print("AUC score: ", roc_auc_score(y_val, custom_pipeline1.predict_proba(X_val)[:, 1]))

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] Unknown parameter: silent
AUC score:  0.8653907412154154


### Conclusion: the OneHotEncoder did not improve the result<br>

### Now with proper mean-encoding as done in catboost<br>

we first transform the train and test set for the early stopping:

In [13]:
preprocessor2 = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical),
        ('mean_enc', ValidatedStackedMeanEncoder(), categorical)
        ])

new_X = preprocessor2.fit_transform(X_train, y_train)
new_X_val = preprocessor2.transform(X_val)

in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in else of transform


In [14]:
custom_pipeline = make_pipeline(
            preprocessor2,
            model
            )
from sklearn.model_selection import StratifiedKFold
custom_pipeline.fit(X_train, y_train, lgbmclassifier__eval_set=[(new_X, y_train), (new_X_val, y_val)])
#custom_pipeline.predict(test)
print("model score: %.3f" % custom_pipeline.score(X_val, y_val))

in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] Unknown parameter: silent
[LightGBM] [Warning] early_stopping_round is set=100, early_stopping_rounds=100 will be ignored. Current value: early_stopping_round=100
[LightGBM] [Warning] Unknown parameter: silent
[LightGBM] [Info] Number of positive: 24687, number of negative: 1528
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015304 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4079
[LightGBM] [Info] Number of data points in the train set: 26215, number of used features: 18
[LightGBM] [Warning] Unknown parameter: silent
[LightGBM] [Warning] early_stopping_round is set=100, early_stopping_rounds=100 will be ignored. Current value: early_stopping_round=100
[LightGBM] [Warning] Unknown parameter: silent
[LightGBM] [Warning] early_stopping_round is set=100, early_stopping_rounds=100 will be ignored. Current value: ear

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [15]:
from sklearn.metrics import classification_report,confusion_matrix, precision_recall_curve, roc_curve, roc_auc_score, log_loss

print(classification_report(y_val, custom_pipeline.predict(X_val)))
print("AUC score: ", roc_auc_score(y_val, custom_pipeline.predict_proba(X_val)[:, 1]))

in else of transform


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] Unknown parameter: silent
              precision    recall  f1-score   support

           0       0.68      0.19      0.30       369
           1       0.95      0.99      0.97      6185

    accuracy                           0.95      6554
   macro avg       0.82      0.59      0.64      6554
weighted avg       0.94      0.95      0.94      6554

in else of transform


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] Unknown parameter: silent
AUC score:  0.8687485896685967


#Exercise: Can we add some Hyper-Parameter Tuning as we did for the Regression example?
Hint: use



1.   `GridSearchCV`
2.   `'lgbmclassifier__learning_rate': 10**(np.linspace(-2, 0.1, 5))`
3.   `'lgbmclassifier__max_depth': [12, 20, 25]`
4.   `scoring = 'roc_auc'`
5.   `refit=True`



In [33]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline

clf = GridSearchCV(
        make_pipeline(
            preprocessor2,
            model
            ),
            param_grid={'lgbmclassifier__learning_rate': 10**(np.linspace(-2, 0.1, 5)),
                        'lgbmclassifier__max_depth': [12, 20, 25]},
         cv=5, refit=True, scoring = 'roc_auc'
)



In [34]:

clf.fit(X_train, y_train, lgbmclassifier__eval_set=[(new_X, y_train), (new_X_val, y_val)])

in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
[LightGBM] [Warning] Unknown parameter: silent
[LightGBM] [Warning] early_stopping_round is set=100, early_stopping_rounds=100 will be ignored. Current value: early_stopping_round=100


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] Unknown parameter: silent
[LightGBM] [Info] Number of positive: 19749, number of negative: 1223
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004868 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4086
[LightGBM] [Info] Number of data points in the train set: 20972, number of used features: 18
[LightGBM] [Warning] Unknown parameter: silent
[LightGBM] [Warning] early_stopping_round is set=100, early_stopping_rounds=100 will be ignored. Current value: early_stopping_round=100
[LightGBM] [Warning] Unknown parameter: silent
[LightGBM] [Warning] early_stopping_round is set=100, early_stopping_rounds=100 will be ignored. Current value: early_stopping_round=100
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.941684 -> initscore=2.781796
[LightGBM] [Info] Start training from score 2.781796
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration i

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] Unknown parameter: silent
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


... (294 outputs left out)

[LightGBM] [Warning] Unknown parameter: silent
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform
in fit_transform


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] Unknown parameter: silent
[LightGBM] [Warning] early_stopping_round is set=100, early_stopping_rounds=100 will be ignored. Current value: early_stopping_round=100
[LightGBM] [Warning] Unknown parameter: silent
[LightGBM] [Info] Number of positive: 24687, number of negative: 1528
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.016083 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4079
[LightGBM] [Info] Number of data points in the train set: 26215, number of used features: 18
[LightGBM] [Warning] Unknown parameter: silent
[LightGBM] [Warning] early_stopping_round is set=100, early_stopping_rounds=100 will be ignored. Current value: early_stopping_round=100
[LightGBM] [Warning] Unknown parameter: silent
[LightGBM] [Warning] early_stopping_round is set=100, early_stopping_rounds=100 will be ignored. Current value: early_stopping_round=100
[LightGBM] [Info] [binary:BoostFromScore]:

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('columntransformer',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         []),
                                                                        ('mean_enc',
                                                                         ValidatedStackedMeanEncoder(),
                                                                         Index(['RESOURCE', 'MGR_ID', 'ROLE_ROLLUP_1', 'ROLE_ROLLUP_2', 'ROLE_DEPTNAME',
       'ROLE_TITLE', 'ROLE_FAMILY_DESC', 'R...
                                        LGBMClassifier(early_stopping_rounds=100,
                                                       is_unbalance=False,
                                                       learning_rate=0.02,
                                                       metric='auc',
                                                       n_estimators=5000,
                                                       n_jobs=-1,
                                                       objective='binary',
                                                       random_state=42,
                                                       scale_pos_weight=1.0,
                                                       silent=True,
                                                       subsample_for_bin=20000,
                                                       verbose=1))]),
             param_grid={'lgbmclassifier__learning_rate': array([0.01      , 0.03349654, 0.11220185, 0.3758374 , 1.25892541]),
                         'lgbmclassifier__max_depth': [12, 20, 25]},
             scoring='roc_auc')

In [35]:
clf.best_params_

{'lgbmclassifier__learning_rate': 0.03349654391578276,
 'lgbmclassifier__max_depth': 12}

In [36]:
clf.best_score_

0.8462150804907544

In [37]:
print(classification_report(y_val, clf.predict(X_val)))
print("AUC score: ", roc_auc_score(y_val, custom_pipeline.predict_proba(X_val)[:, 1]))

in else of transform


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] Unknown parameter: silent
              precision    recall  f1-score   support

           0       0.69      0.23      0.35       369
           1       0.96      0.99      0.97      6185

    accuracy                           0.95      6554
   macro avg       0.82      0.61      0.66      6554
weighted avg       0.94      0.95      0.94      6554

in else of transform
[LightGBM] [Warning] Unknown parameter: silent
AUC score:  0.8687485896685967


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
